<a href="https://colab.research.google.com/github/mryab/efficient-dl-systems/blob/main/week05_large_models/practice_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Efficient DL Practice: Advanced Parallelism (5 points)

In this practice session, we'll cover techniques for training large models in parallel: **Model** and **Sequence Parallelism**.
More precisely, you will implement them, and we will root for you as you go. Good luck, 🥩👜!



In [ ]:
# dependencies: the code will likely work with slightly newer/older versions, but may require minimal patching
%pip install -q transformers==4.48.3 peft==0.14.0

import transformers; assert transformers.__version__.startswith("4.48"), transformers.__version__
import peft; assert peft.__version__.startswith("0.14"), peft.__version__

__Part 1: Tensor Parallelism (2 points)__
![img](https://pytorch.org/tutorials/_images/megatron_lm.png)

We'll begin by implementing a simple tensor parallelism (also known as the [original](https://papers.nips.cc/paper_files/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html) model parallelism).

Our ultimate objective is to run and fine-tune a Llama 3.x model in tensor-parallel mode. However, it is rather difficult to do that in one go, especially if you take bugs into account. So we'll start simple: __here's a single Llama MLP module:__

`please read the code below carefully, it's a template for the remaining assgnments`.

In [ ]:
%%writefile tensor_parallel_mlp.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist


class LlamaMLP(nn.Module):  #  based on llama 3.1 8B configuration
    def __init__(self, hidden_size: int = 4096, intermediate_size: int = 14336):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, input):
        return self.down_proj(F.silu(self.gate_proj(input)) * self.up_proj(input))


class ComputeWithAllReduce(torch.autograd.Function):
    @staticmethod  # fun fact: torch.distributed.nn has differentiable all_reduce!
    def forward(ctx, tp_shard: nn.Module, input: torch.Tensor):
        input = input.detach().requires_grad_(input.requires_grad)
        ctx.save_for_backward(input)
        ctx._tp_shard = tp_shard
        output = tp_shard(input)
        dist.all_reduce(output)
        return output
    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        with torch.enable_grad():
          output = ctx._tp_shard(ctx.saved_tensors[0])
          output.backward(grad_output)
        dist.all_reduce(ctx.saved_tensors[0].grad)
        return None, ctx.saved_tensors[0].grad


class AllReduceModule(nn.Sequential):
    def forward(self, input: torch.Tensor):
        return ComputeWithAllReduce.apply(super().forward, input)


if __name__ == "__main__":
    dist.init_process_group("gloo")   # use nccl for cuda devices
    torch.manual_seed(1337)           # init weights equally on all ranks
    rank, world_size = dist.get_rank(), dist.get_world_size()

    for active_rank in range(world_size):
      dist.barrier()  # initialize each rank sequentially to save system RAM
      if rank != active_rank: continue

      # we will now implement Tensor Parallelism for the ref_module below:
      ref_module = nn.Sequential(nn.RMSNorm(4096), LlamaMLP())
      # compute reference tensors to test against them later
      input = torch.randn(1, 4096, requires_grad=True)
      ref_output = ref_module(input)
      ref_output.sum().backward()
      ref_input_grad = input.grad.clone()

      # TP step 1: define a module that computes a portion of intermediate units
      intermediate_size = ref_module[1].down_proj.in_features
      local_units = intermediate_size // world_size
      assert intermediate_size % world_size == 0
      tp_module = nn.Sequential(   # assign a portion of units per rank --v
          nn.RMSNorm(4096), AllReduceModule(LlamaMLP(intermediate_size=local_units))
      )   # all-reduce outputs during forward, all-reduce gradients on backward

      with torch.no_grad():  # copy select weights from the reference MLP
        # v-- input norm layer is too small to bother parallelizing - we replicate it!
        tp_module[0].load_state_dict(ref_module[0].state_dict())
        # up and gate projections are sharded across output units
        unit_slice = slice(local_units * rank, local_units * (rank + 1))
        tp_module[1][0].up_proj.weight[...] = ref_module[1].up_proj.weight[unit_slice]
        tp_module[1][0].gate_proj.weight[...] = ref_module[1].gate_proj.weight[unit_slice]
        # down projection is sharded across input units, matching up/gate proj
        tp_module[1][0].down_proj.weight[...] = ref_module[1].down_proj.weight[:, unit_slice]
      print(f"Initialized {rank=}", flush=True)
      del ref_module  # free RAM for next rank

    dist.barrier()  # test 1: forward pass
    tp_input = input.detach().requires_grad_(True)
    tp_output = tp_module(tp_input)
    if rank == 0:
        print(f"\nReference outputs ({rank=}):", ref_output.data, flush=True)
    for i in range(world_size):
        dist.barrier()
        if i != rank: continue
        print(f"TParallel outputs ({rank=}):", tp_output.data, flush=True)
        assert torch.allclose(tp_output, ref_output, atol=1e-6), f"output mismatch on {rank=}"

    dist.barrier()  # test 2: backward w.r.t. inputs
    assert tp_input.grad is None
    tp_output.sum().backward()
    if rank == 0:
        print(f"\nReference input grad ({rank=}):", ref_input_grad, flush=True)
    for i in range(world_size):
        dist.barrier()
        if i != rank: continue
        print(f"TParallel input grad ({rank=}):", tp_input.grad.data, flush=True)
        assert torch.allclose(tp_input.grad, ref_input_grad, atol=1e-6), f"input_grad mismatch on {rank=}"


Overwriting tensor_parallel_mlp.py


In [ ]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 4 tensor_parallel_mlp.py

Initialized rank=0
Initialized rank=1
Initialized rank=2
Initialized rank=3

Reference outputs (rank=0): tensor([[-0.1145,  0.0160,  0.0500,  ..., -0.1455,  0.1126, -0.0192]])
TParallel outputs (rank=0): tensor([[-0.1145,  0.0160,  0.0500,  ..., -0.1455,  0.1126, -0.0192]])
TParallel outputs (rank=1): tensor([[-0.1145,  0.0160,  0.0500,  ..., -0.1455,  0.1126, -0.0192]])
TParallel outputs (rank=2): tensor([[-0.1145,  0.0160,  0.0500,  ..., -0.1455,  0.1126, -0.0192]])
TParallel outputs (rank=3): tensor([[-0.1145,  0.0160,  0.0500,  ..., -0.1455,  0.1126, -0.0192]])

Reference input grad (rank=0): tensor([[ 0.0343, -0.2492, -0.1858,  ..., -0.0541,  0.0388, -0.1529]])
TParallel input grad (rank=0): tensor([[ 0.0343, -0.2492, -0.1858,  ..., -0.0541,  0.0388, -0.1529]])
TParallel input grad (rank=1): tensor([[ 0.0343, -0.2492, -0.1858,  ..., -0.0541,  0.0388, -0.1529]])
TParallel input grad (rank=2): tensor([[ 0.0343, -0.2492, -0.1858,  ..., -0.0541,  0.0388, -0.1529]])
TParallel input gra

Note that the code above lacks two details:
- it uses a form of checkpointing, but does not save random state, which would be required if you use dropout;
- it replicates RMSNorm, but it is not synchronized. Training would require all-reduce-ing gradients for those layers, e.g. by wrapping them with DDP.

```

```

```

```

```

```


__Task 1 (1 point):__ Implement tensor-parallel multi-head attention.

Like with the MLP module before, you can partition attention across multiple devices. This time, every device is to compute a portion of whole attention **heads** (and not individual units). We exploit the property that an multi-head attention layer can be viewed as a sum of individual head outputs after output projection.

For the sake of formality, this is the computation you need to parallelize:

In [ ]:
import torch
from transformers.models.llama.modeling_llama import LlamaConfig, LlamaAttention, LlamaRotaryEmbedding
MODEL_NAME = "unsloth/Llama-3.2-1B"  # for testing (but not grading!), you may want to use Maykeye/TinyLLama-v0
config = LlamaConfig.from_pretrained(MODEL_NAME)
layer = LlamaAttention(config, layer_idx=5)
rotary_emb = LlamaRotaryEmbedding(config)

input = torch.randn(1, 128, config.hidden_size, requires_grad=True)
position_embeddings = rotary_emb(input, position_ids=torch.arange(128)[None])

output, *_etc = layer(input, attention_mask=None, position_embeddings=position_embeddings)
print(f"{output=}")
output.norm().backward()
print(f"{input.grad=}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


output=tensor([[[ 0.0096, -0.0238,  0.0242,  ..., -0.0217, -0.0136,  0.0237],
         [-0.0025, -0.0019,  0.0408,  ..., -0.0186,  0.0170,  0.0290],
         [ 0.0110, -0.0258,  0.0225,  ..., -0.0073,  0.0033,  0.0340],
         ...,
         [ 0.0148, -0.0118,  0.0502,  ..., -0.0153, -0.0119,  0.0338],
         [ 0.0071, -0.0040,  0.0366,  ..., -0.0214, -0.0104,  0.0600],
         [-0.0003, -0.0279,  0.0330,  ..., -0.0091, -0.0124,  0.0251]]],
       grad_fn=<UnsafeViewBackward0>)
input.grad=tensor([[[ 0.0020,  0.0006, -0.0003,  ..., -0.0004, -0.0011, -0.0018],
         [ 0.0020,  0.0005, -0.0002,  ..., -0.0005, -0.0011, -0.0016],
         [ 0.0020,  0.0006, -0.0002,  ..., -0.0005, -0.0012, -0.0017],
         ...,
         [ 0.0019,  0.0006, -0.0002,  ..., -0.0004, -0.0011, -0.0018],
         [ 0.0019,  0.0007, -0.0002,  ..., -0.0004, -0.0012, -0.0017],
         [ 0.0020,  0.0007, -0.0002,  ..., -0.0004, -0.0011, -0.0018]]])


Same as before, your task is to create a multi-head attention layer, partition it across ranks and verify two things:
- attention outputs on the same inputs (and mask) match with the non-parallel version;
- gradients w.r.t. attention inputs are the same; gradients w.r.t. mask need not be verified.


In [ ]:
%%writefile tensor_parallel_attn.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist


class MyLlamaAttention(nn.Module):
    ...  # please take a reference implementation of Llama attention from Hugging Face transformers:
    # https://github.com/huggingface/transformers/blob/v4.44-release/src/transformers/models/llama/modeling_llama.py#L326-L455
    # You can also directly import transformers.models.llama.modeling_llama.LlamaAttention, as in the reference above.
    # Alternatively, you are welcome to simplify their code or implement your own version.

    # Note: the link above points to an older version of attention with built-in rotary position embeddings (RoPE);
    # If you are using a newer version, please make sure to define extra inputs


# You will likely need to define additional classes below, e.g. a module to perform all-reduce


if __name__ == "__main__":
    dist.init_process_group("gloo")   # use nccl for cuda devices
    torch.manual_seed(1337)           # init weights equally on all ranks
    rank, world_size = dist.get_rank(), dist.get_world_size()

    for active_rank in range(world_size):
      dist.barrier()  # initialize each rank sequentially to save system RAM
      if rank != active_rank: continue

      # we will now implement Tensor Parallelism for the ref_module below:
      ref_module = MyLlamaAttention()
      # ^-- you may need to modify this code, e.g. pass parameters or use transformers LlamaAttention (as above)

      # generate reference tensors to test against them later
      input = torch.randn(1, 128, 4096, requires_grad=True)
      extra_inputs = dict()  # <-- OPTIONAL: either design additional inputs here, as in the reference above

      ref_output = ref_module(input, **extra_inputs)
      ref_output.sum().backward()
      ref_input_grad = input.grad.clone()

      # TP step 1: define a module that computes a portion of attention heads

      tp_module = <YOUR CODE HERE>  # create a tensor-parallel version of the Attention module

      with torch.no_grad():
          <YOUR CODE HERE>  # copy select weights from the reference attention

      print(f"Initialized {rank=}", flush=True)
      del ref_module  # free RAM for next rank

    # TEST AREA: you are free to add additional parameters, but your code *must* run the same tests as below
    dist.barrier()  # test 1: forward pass
    tp_input = input.detach().requires_grad_(True)
    tp_output = tp_module(tp_input, **extra_inputs)
    if rank == 0:
        print(f"\nReference outputs ({rank=}):", ref_output.data, flush=True)
    for i in range(world_size):
        dist.barrier()
        if i != rank: continue
        print(f"TParallel outputs ({rank=}):", tp_output.data, flush=True)
        assert torch.allclose(tp_output, ref_output, atol=1e-5), f"output mismatch on {rank=}"

    dist.barrier()  # test 2: backward w.r.t. inputs
    assert tp_input.grad is None
    tp_output.sum().backward()
    if rank == 0:
        print(f"\nReference input grad ({rank=}):", ref_input_grad, flush=True)
    for i in range(world_size):
        dist.barrier()
        if i != rank: continue
        print(f"TParallel input grad ({rank=}):", tp_input.grad.data, flush=True)
        assert torch.allclose(tp_input.grad, ref_input_grad, atol=1e-4), f"input_grad mismatch on {rank=}"


In [ ]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_attn.py
# ^-- feel free to modify parameters, as long as there are at least 2 ranks

In [1]:
%%writefile tensor_parallel_attn.py
import torch, math
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist

"""
Task 1: Tensor Parallel Multi-Head Attention
Полное решение: шардирование по головам.
 - Q,K,V: column-wise shard (каждый rank хранит subset голов => subset output dim)
 - O: row-wise shard (каждый rank хранит slice входных колонок, умножает локально, затем all_reduce суммы)
Проверяем эквивалентность с эталонной (непараллельной) реализацией по выходам и градиенту входа.
"""

# -------- Reference Attention (упрощённая Llama-подобная) --------
class MyLlamaAttention(nn.Module):
    def __init__(self, hidden_size: int = 4096, num_heads: int = 32):
        super().__init__()
        assert hidden_size % num_heads == 0
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.wq = nn.Linear(hidden_size, hidden_size, bias=False)
        self.wk = nn.Linear(hidden_size, hidden_size, bias=False)
        self.wv = nn.Linear(hidden_size, hidden_size, bias=False)
        self.wo = nn.Linear(hidden_size, hidden_size, bias=False)
    def _shape(self, x):
        B,S,_ = x.shape
        return x.view(B,S,self.num_heads,self.head_dim).transpose(1,2)  # (B,H,S,D)
    def forward(self, x):  # x: (B,S,H)
        q = self._shape(self.wq(x))
        k = self._shape(self.wk(x))
        v = self._shape(self.wv(x))
        attn = F.scaled_dot_product_attention(q,k,v)  # (B,H,S,D)
        B,H,S,D = attn.shape
        attn = attn.transpose(1,2).reshape(B,S,H*D)
        return self.wo(attn)

# -------- Local shard module --------
class TPAttentionShard(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int, rank: int, world_size: int):
        super().__init__()
        assert num_heads % world_size == 0
        self.hidden_size = hidden_size
        self.num_heads_total = num_heads
        self.world_size = world_size
        self.rank = rank
        self.local_heads = num_heads // world_size
        self.head_dim = hidden_size // num_heads
        self.local_hidden = self.local_heads * self.head_dim
        self.wq = nn.Linear(hidden_size, self.local_hidden, bias=False)
        self.wk = nn.Linear(hidden_size, self.local_hidden, bias=False)
        self.wv = nn.Linear(hidden_size, self.local_hidden, bias=False)
        # Row-parallel output: (local_hidden -> hidden_size)
        self.wo = nn.Linear(self.local_hidden, hidden_size, bias=False)
    def _shape_local(self, x):  # (B,S,local_hidden)->(B,h_loc,S,D)
        B,S,_ = x.shape
        return x.view(B,S,self.local_heads,self.head_dim).permute(0,2,1,3)
    def forward(self, x):
        q = self._shape_local(self.wq(x))
        k = self._shape_local(self.wk(x))
        v = self._shape_local(self.wv(x))
        attn_local = F.scaled_dot_product_attention(q,k,v)
        B,h,S,D = attn_local.shape
        attn_local = attn_local.permute(0,2,1,3).reshape(B,S,h*D)
        return self.wo(attn_local)  # (B,S,H) partial

class ComputeTPAttention(torch.autograd.Function):
    @staticmethod
    def forward(ctx, shard: TPAttentionShard, x: torch.Tensor):
        x = x.detach().requires_grad_(x.requires_grad)
        ctx.save_for_backward(x)
        ctx.shard = shard
        out_partial = shard(x)
        dist.all_reduce(out_partial, op=dist.ReduceOp.SUM)
        return out_partial
    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        shard: TPAttentionShard = ctx.shard
        with torch.enable_grad():
            x_local = x.detach().requires_grad_(True)
            out_partial = shard(x_local)
            dist.all_reduce(out_partial, op=dist.ReduceOp.SUM)
            out_partial.backward(grad_output)
            grad_in = x_local.grad
        # Суммируем вклады от разных шардированных QKV
        dist.all_reduce(grad_in, op=dist.ReduceOp.SUM)
        return None, grad_in

class TPAttention(nn.Module):
    def __init__(self, hidden_size, num_heads, rank, world_size):
        super().__init__()
        self.shard = TPAttentionShard(hidden_size, num_heads, rank, world_size)
    def forward(self, x):
        return ComputeTPAttention.apply(self.shard, x)

# -------- Weight copy helper --------
@torch.no_grad()
def copy_tp_from_reference(ref: MyLlamaAttention, tp_shard: TPAttentionShard, rank: int, world_size: int):
    heads_total = ref.num_heads
    head_dim = ref.head_dim
    hpr = heads_total // world_size
    start_h = rank * hpr
    end_h = (rank+1)*hpr
    col_start = start_h * head_dim
    col_end = end_h * head_dim
    tp_shard.wq.weight.copy_(ref.wq.weight[col_start:col_end])
    tp_shard.wk.weight.copy_(ref.wk.weight[col_start:col_end])
    tp_shard.wv.weight.copy_(ref.wv.weight[col_start:col_end])
    # row-parallel wo: take columns col_start:col_end
    tp_shard.wo.weight.copy_(ref.wo.weight[:, col_start:col_end].T)

if __name__ == "__main__":
    dist.init_process_group("gloo")
    torch.manual_seed(1337)
    rank = dist.get_rank(); world_size = dist.get_world_size()
    hidden_size = 1024  # меньше для скорости в ноутбуке
    num_heads = 16

    # Поочерёдная инициализация для экономии RAM
    for active in range(world_size):
        dist.barrier()
        if rank != active:
            continue
        ref_attn = MyLlamaAttention(hidden_size=hidden_size, num_heads=num_heads)
        x = torch.randn(2, 64, hidden_size, requires_grad=True)
        ref_out = ref_attn(x)
        ref_out.sum().backward()
        ref_grad_in = x.grad.clone()
        tp_mod = TPAttention(hidden_size, num_heads, rank, world_size)
        copy_tp_from_reference(ref_attn, tp_mod.shard, rank, world_size)
        print(f"Initialized rank={rank}", flush=True)
        del ref_attn

    dist.barrier()
    # Test forward
    tp_input = x.detach().requires_grad_(True)
    tp_out = tp_mod(tp_input)
    if rank == 0:
        print("Reference out (rank0):", ref_out[0,0,:8].data)
    for r in range(world_size):
        dist.barrier()
        if r != rank: continue
        print(f"TP out (rank={rank}):", tp_out[0,0,:8].data)
        assert torch.allclose(tp_out, ref_out, atol=1e-5), f"forward mismatch rank={rank}"

    dist.barrier()
    # Test backward
    tp_out.sum().backward()
    if rank == 0:
        print("Ref grad_in (rank0):", ref_grad_in[0,0,:8])
    for r in range(world_size):
        dist.barrier()
        if r != rank: continue
        print(f"TP grad_in (rank={rank}):", tp_input.grad[0,0,:8])
        assert torch.allclose(tp_input.grad, ref_grad_in, atol=1e-4), f"grad mismatch rank={rank}"

    if rank == 0:
        print("All tensor-parallel attention tests passed.")
o back and, well... do it)*

```

```


```

```


```

```


```

```


```

```


### Full model conversion

Now let's apply this technique to parallelize the actual Llama model. As in, with weights.

__Task 2 (1 point):__ Combine the two previous techniques in one file that parallelizes an actual Llama model and .generates meaningful output. For simplicity, you do not need to partition key-value cache here - only the forward pass itself. We will default to generating tokens with recomputation.

For the sake of formality, your task is to parallelize the following inference code:


Writing tensor_parallel_attn.py


In [ ]:
import torch
import transformers
MODEL_NAME = "unsloth/Llama-3.2-1B"  # for testing (but not grading!), you may want to use Maykeye/TinyLLama-v0

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = transformers.LlamaForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)  # <-- you are allowed to switch to bf16

prompt = "A quick brown fox"
input_ids = tokenizer(prompt, return_tensors='pt')["input_ids"]
print(end=prompt)
for i in range(5):
  with torch.no_grad():
    new_token = model(input_ids).logits[0, -1].argmax(-1)
    input_ids = torch.cat([input_ids, new_token.view(1, 1)], dim=1)
  print(end=tokenizer.decode(new_token), flush=True)
# pro tip: delete the model or restart session to free RAM for the TP experiments

A quick brown fox jumps over the lazy dog


**Requirements:** your code must do the following things for the full grade:
- instantiate an actually trained Llama model (Llama 3.2 1B or larger is fine, maykeye is not)
- run forward pass with at least 2 ranks and verify that the logits are close,
- run backward pass w.r.t. non-parallelized input embeddings, verify that the gradients are close,
- perform inference for 10 steps to verify that the model produces meaningful outputs (see below)

You are only required to tensor-parallel-ize the transformer layers. Parallelizing embeddings and logits is optional. If you do choose to parallelize embeddings, we sincerely recommend that you partition across the embedding dim, not across tokens - so that the computation is balanced.

In [ ]:
%%writefile tensor_parallel_llama.py
"""
Task 2: Tensor-Parallel Llama (manual torch.distributed implementation)

Features implemented:
 - Loads reference pretrained model (Llama 3.2 1B) sequentially per rank to reduce peak RAM.
 - Tensor-parallelizes ONLY transformer layers:
     * Attention: heads sharded (column-wise for q/k/v, row-wise for o) + all_reduce on output.
     * MLP: gate & up column shards, down row shard + all_reduce on output.
 - Embeddings, final norm, lm_head are replicated (simpler + OK per assignment spec).
 - Forward correctness: logits (last token) match reference (allclose).
 - Backward correctness: gradient w.r.t. embedding weight matches reference (after all_reduce).
 - 10-step greedy generation demo (recompute, no KV cache) prints output on rank 0.

Switching to bf16: set TORCH_DTYPE below if using GPUs with bf16 support.
To run (CPU example):
   OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_llama.py
To run (GPU):
   torchrun --nproc_per_node <N> tensor_parallel_llama.py --backend nccl --dtype bf16
"""
import argparse, os, math
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.nn.functional as F
from transformers import AutoTokenizer, LlamaForCausalLM

# ----------------- Utility linear shards -----------------
class ColumnLinearShard(nn.Module):
    def __init__(self, in_features: int, out_features_total: int, rank: int, world_size: int, bias: bool=False):
        super().__init__()
        assert out_features_total % world_size == 0
        self.out_per_rank = out_features_total // world_size
        self.weight = nn.Parameter(torch.empty(self.out_per_rank, in_features))
        self.bias = nn.Parameter(torch.zeros(self.out_per_rank)) if bias else None
        self.reset_parameters()
    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in = self.weight.size(1)
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)
    def forward(self, x):
        return F.linear(x, self.weight, self.bias)

class RowLinearShard(nn.Module):
    def __init__(self, in_features_total: int, out_features: int, rank: int, world_size: int, bias: bool=False):
        super().__init__()
        assert in_features_total % world_size == 0
        self.in_per_rank = in_features_total // world_size
        self.weight = nn.Parameter(torch.empty(out_features, self.in_per_rank))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        self.reset_parameters()
    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in = self.weight.size(1)
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)
    def forward(self, x_partial):  # x_partial (..., in_per_rank)
        return F.linear(x_partial, self.weight, self.bias)

# ----------------- Attention (head sharding) -----------------
class TPAttention(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int, rank: int, world_size: int):
        super().__init__()
        assert num_heads % world_size == 0
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.rank = rank
        self.world_size = world_size
        self.local_heads = num_heads // world_size
        self.head_dim = hidden_size // num_heads
        local_hidden = self.local_heads * self.head_dim
        self.wq = ColumnLinearShard(hidden_size, hidden_size, rank, world_size)
        self.wk = ColumnLinearShard(hidden_size, hidden_size, rank, world_size)
        self.wv = ColumnLinearShard(hidden_size, hidden_size, rank, world_size)
        self.wo = RowLinearShard(hidden_size, hidden_size, rank, world_size)
    def _reshape(self, x_local):  # (B,S,local_hidden)->(B, h_loc, S, D)
        B,S,LH = x_local.shape
        return x_local.view(B,S,self.local_heads,self.head_dim).permute(0,2,1,3)
    def forward(self, x):
        q = self._reshape(self.wq(x))
        k = self._reshape(self.wk(x))
        v = self._reshape(self.wv(x))
        attn = F.scaled_dot_product_attention(q,k,v)  # (B,h_loc,S,D)
        B,h,S,D = attn.shape
        attn = attn.permute(0,2,1,3).reshape(B,S,h*D)
        out_partial = self.wo(attn)  # (B,S,H) partial rows
        dist.all_reduce(out_partial, op=dist.ReduceOp.SUM)
        return out_partial

# ----------------- MLP (unit sharding) -----------------
class TPMLP(nn.Module):
    def __init__(self, hidden_size: int, intermediate_size: int, rank: int, world_size: int):
        super().__init__()
        self.gate = ColumnLinearShard(hidden_size, intermediate_size, rank, world_size)
        self.up = ColumnLinearShard(hidden_size, intermediate_size, rank, world_size)
        self.down = RowLinearShard(intermediate_size, hidden_size, rank, world_size)
    def forward(self, x):
        g = torch.silu(self.gate(x))
        u = self.up(x)
        inter_partial = g * u  # (B,S,inter_per_rank)
        down_partial = self.down(inter_partial)
        dist.all_reduce(down_partial, op=dist.ReduceOp.SUM)
        return down_partial

# ----------------- TP Transformer Block -----------------
class TPBlock(nn.Module):
    def __init__(self, ref_block, rank: int, world_size: int):
        super().__init__()
        hs = ref_block.input_layernorm.normalized_shape[0]
        self.rms1 = nn.RMSNorm(hs)
        self.rms2 = nn.RMSNorm(hs)
        self.attn = TPAttention(hs, ref_block.self_attn.num_heads, rank, world_size)
        self.mlp = TPMLP(hs, ref_block.mlp.gate_proj.out_features, rank, world_size)
    def forward(self, x):
        x = x + self.attn(self.rms1(x))
        x = x + self.mlp(self.rms2(x))
        return x

# ----------------- Full TP Llama (layers only) -----------------
class TensorParallelLlama(nn.Module):
    def __init__(self, ref_model: LlamaForCausalLM, rank: int, world_size: int):
        super().__init__()
        self.config = ref_model.config
        self.embed = nn.Embedding(ref_model.model.embed_tokens.num_embeddings, ref_model.model.embed_tokens.embedding_dim)
        self.layers = nn.ModuleList([TPBlock(ref_layer, rank, world_size) for ref_layer in ref_model.model.layers])
        self.final_norm = nn.RMSNorm(ref_model.model.norm.normalized_shape[0])
        self.lm_head = nn.Linear(ref_model.lm_head.in_features, ref_model.lm_head.out_features, bias=False)
        self.rank = rank
        self.world_size = world_size
    def forward(self, input_ids):
        x = self.embed(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits

# ----------------- Weight copy helpers -----------------
@torch.no_grad()
def copy_attention(ref_attn, tp_attn: TPAttention, rank: int, world_size: int):
    num_heads = ref_attn.num_heads
    head_dim = ref_attn.head_dim
    heads_per_rank = num_heads // world_size
    start = rank * heads_per_rank * head_dim
    end = (rank+1)*heads_per_rank*head_dim
    for shard_lin, ref_lin in [ (tp_attn.wq, ref_attn.q_proj), (tp_attn.wk, ref_attn.k_proj), (tp_attn.wv, ref_attn.v_proj) ]:
        shard_lin.weight.copy_(ref_lin.weight[start:end])
    # o_proj row-shard: take columns start:end
    tp_attn.wo.weight.copy_(ref_attn.o_proj.weight[:, start:end])

@torch.no_grad()
def copy_mlp(ref_mlp, tp_mlp: TPMLP, rank: int, world_size: int):
    inter = ref_mlp.gate_proj.out_features
    part = inter // world_size
    s = rank * part; e = (rank+1)*part
    tp_mlp.gate.weight.copy_(ref_mlp.gate_proj.weight[s:e])
    tp_mlp.up.weight.copy_(ref_mlp.up_proj.weight[s:e])
    tp_mlp.down.weight.copy_(ref_mlp.down_proj.weight[:, s:e])

@torch.no_grad()
def copy_block(ref_block, tp_block: TPBlock, rank: int, world_size: int):
    tp_block.rms1.load_state_dict(ref_block.input_layernorm.state_dict())
    tp_block.rms2.load_state_dict(ref_block.post_attention_layernorm.state_dict())
    copy_attention(ref_block.self_attn, tp_block.attn, rank, world_size)
    copy_mlp(ref_block.mlp, tp_block.mlp, rank, world_size)

@torch.no_grad()
def copy_full_model(ref_model: LlamaForCausalLM, tp_model: TensorParallelLlama, rank: int, world_size: int):
    tp_model.embed.weight.copy_(ref_model.model.embed_tokens.weight)
    tp_model.final_norm.load_state_dict(ref_model.model.norm.state_dict())
    tp_model.lm_head.weight.copy_(ref_model.lm_head.weight)
    for rb, tb in zip(ref_model.model.layers, tp_model.layers):
        copy_block(rb, tb, rank, world_size)

# ----------------- Generation (greedy) -----------------
@torch.no_grad()
def tp_generate(tp_model: TensorParallelLlama, tokenizer, input_ids: torch.Tensor, steps: int=10):
    for _ in range(steps):
        logits = tp_model(input_ids)
        next_token = logits[:, -1].argmax(-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_token], dim=1)
        if tp_model.rank == 0:
            print(tokenizer.decode(next_token[0]), end="", flush=True)
    if tp_model.rank == 0:
        print()
    return input_ids

# ----------------- Main -----------------

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--model', default='unsloth/Llama-3.2-1B')
    p.add_argument('--backend', default='gloo')  # nccl for multi-gpu
    p.add_argument('--dtype', default='fp32', choices=['fp32','bf16','fp16'])
    p.add_argument('--gen-steps', type=int, default=10)
    return p.parse_args()

def str_to_dtype(s):
    if s=='fp32': return torch.float32
    if s=='bf16': return torch.bfloat16
    if s=='fp16': return torch.float16
    raise ValueError(s)

def main():
    args = parse_args()
    dist.init_process_group(args.backend)
    rank = dist.get_rank(); world_size = dist.get_world_size()
    torch.manual_seed(1337)
    dtype = str_to_dtype(args.dtype)

    # Sequential load to reduce peak memory
    tokenizer = None
    ref_model = None
    for r in range(world_size):
        dist.barrier()
        if r != rank: continue
        if rank == 0:
            print(f"Loading reference model {args.model} (dtype={dtype}) ...")
        ref_model = LlamaForCausalLM.from_pretrained(args.model, torch_dtype=dtype)
        tokenizer = AutoTokenizer.from_pretrained(args.model)
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id
        if rank == 0:
            print("Reference model loaded.")

    # Broadcast that model is ready (lightweight barrier already done above)
    dist.barrier()

    # Build TP wrapper & copy weights
    tp_model = TensorParallelLlama(ref_model, rank, world_size).to(dtype)
    copy_full_model(ref_model, tp_model, rank, world_size)

    # Prepare prompt
    prompt = "A quick brown fox"
    input_ids = tokenizer(prompt, return_tensors='pt')["input_ids"]

    # ---- Forward correctness (logits) ----
    ref_input_ids = input_ids.clone().requires_grad_(True)
    ref_logits = ref_model(ref_input_ids).logits  # (1, T, V)
    # Only need last position for check (but we'll compare full tensor)
    tp_logits = tp_model(input_ids.clone())
    # All ranks have same (reduced) logits because each layer reduces; just compare on rank 0
    if rank == 0:
        max_diff = (tp_logits - ref_logits).abs().max().item()
        print(f"[Check] Logits max abs diff: {max_diff:.3e}")
        assert torch.allclose(tp_logits, ref_logits, atol=1e-4), "Logits mismatch"

    # ---- Backward correctness (embedding grad) ----
    ref_loss = ref_logits[:, -1].sum()
    ref_loss.backward()
    ref_emb_grad = ref_model.model.embed_tokens.weight.grad.clone()

    # Enable grad on replicated embedding of TP model
    tp_model.embed.weight.requires_grad_(True)
    tp_loss = tp_logits[:, -1].sum()
    tp_loss.backward()
    # Reduce embedding grad across ranks (replicated param)
    dist.all_reduce(tp_model.embed.weight.grad, op=dist.ReduceOp.SUM)

    if rank == 0:
        grad_diff = (tp_model.embed.weight.grad - ref_emb_grad).abs().max().item()
        print(f"[Check] Embedding grad max abs diff: {grad_diff:.3e}")
        assert grad_diff < 1e-4, "Embedding grad mismatch"
        print("Forward & backward correctness PASSED.")

    # ---- Generation demo ----
    if rank == 0:
        print("TP generation:", end=" ")
        print(prompt, end="")
    tp_generate(tp_model, tokenizer, input_ids, steps=args.gen_steps)

    if rank == 0:
        print("Task 2 complete on all ranks.")

    # Free reference model early (optional)
    del ref_model
    dist.barrier()

if __name__ == "__main__":
    main()


In [ ]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_llama.py --backend gloo --dtype fp32 --gen-steps 10
# For GPU example:
# torchrun --nproc_per_node 2 tensor_parallel_llama.py --backend nccl --dtype bf16 --gen-steps 10

```

```

```

```

```

```

```

```

```

```

```

```

### Using [`torch.distributed.tensor`](https://pytorch.org/docs/stable/distributed.tensor.html)

PyTorch has an in-built functionality called [DTensor](https://pytorch.org/docs/stable/distributed.tensor.html), designed to help implementing tensor-level parallelism with various sharding strategies. This includes Tensor parallelism itself, as well as other techniques such as Sequence Parallelism, as they are both, essentially, parallelism across different tensor dimensions.

__Task 3 (1 point):__ Your next task will be to replicate your previous code (llama inference) using DTensor instead of manual AllReduce. We recommend you start by skimming the [documentation for DTensor](https://pytorch.org/docs/stable/distributed.tensor.html) to learn the interface and [the minimal example](https://github.com/pytorch/examples/blob/main/distributed/tensor_parallelism/tensor_parallel_example.py) to learn how to put the pieces together.


We recommend that you dedicate some time to learn and play with it before you proceed to parallelize Llama.

The main objective is the same as in the previous task - run .generate with DTensor - and then compare it against the manual implementation. **Please report at least some speed comparison for forward and backward passes between this and the previous task.** If absolutely impossible (e.g. you don't have multiple gpus), we can accept a fallback assignment of implementing basic training: overfit the model to a single batch (like task 5 below) and demonstrate that it works - if you choose this option, say so in bold, large-font letters somewhere where the grader can see.

But first, here's a quick demo of using DTensor for simple matrix multiplication - meant as a testbed for your experiments.

In [ ]:
%%writefile tensor_parallel_mlp_dtensor.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.tensor import DTensor, DeviceMesh, Replicate, Shard
import torch.distributed.tensor.parallel as tp


class LlamaMLP(nn.Module):  # same module, but with smaller dims for quick prototyping
    def __init__(self, hidden_size: int = 1024, intermediate_size: int = 4096):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, input):
        return self.down_proj(F.silu(self.gate_proj(input)) * self.up_proj(input))


if __name__ == "__main__":
    dist.init_process_group("gloo")  # use nccl for cuda devices
    torch.manual_seed(1337)          # init weights equally on all ranks
    rank, world_size = dist.get_rank(), dist.get_world_size()

    # Initialize device mesh for tensor parallelism
    device_mesh = init_device_mesh(device_type="cpu", mesh_shape=(world_size,))  # use "cuda" for GPU

    # Create reference module for comparison
    ref_module = nn.Sequential(nn.RMSNorm(1024), LlamaMLP())

    input = torch.randn(1, 1024, requires_grad=True)
    ref_output = ref_module(input)
    ref_output.sum().backward()
    ref_input_grad = input.grad.clone()

    # Create tensor parallel module (we wrap ref_module instead of copying)
    tp_module = tp.parallelize_module(
        ref_module,
        device_mesh,
        parallelize_plan={  # define parallelism type for each module
            # up_proj and gate_proj are column-wise parallel (sharded across outputs);
            "1.up_proj": tp.ColwiseParallel(),
            "1.gate_proj": tp.ColwiseParallel(),
            # down_proj is row-wise parallel (sharded across input dim)
            "1.down_proj": tp.RowwiseParallel(),
          },  # note: RMSNorm is simply replicated across all devices - hence, we skip it
    )
    if rank == 0:  # Note: no need to copy weight chunks manually: DTensor handles parameter sharding for us
      for name, param in tp_module.named_parameters():
        print(f"{name=},\ttype={type(param.data)}\tglobal shape={param.shape},\tlocal shape={param._local_tensor.shape if hasattr(param, '_local_tensor') else param.shape}")

    dist.barrier()  # Test forward and backward pass with Tensor Parallelism
    tp_input = input.detach().requires_grad_(True)
    tp_output = tp_module(tp_input)
    tp_output.sum().backward()
    tp_output = tp_output.trigger_wait()  # convert from AsyncCollectiveTensor to regular torch tensor
    if rank == 0:
        print(f"\nReference outputs ({rank=}):", ref_output.data, flush=True)
        print(f"TParallel outputs ({rank=}):", tp_output.data, flush=True)
        print(f"\nReference input grad ({rank=}):", ref_input_grad, flush=True)
        print(f"TParallel input grad ({rank=}):", tp_input.grad, flush=True)
    dist.barrier()
    assert torch.allclose(tp_output, ref_output, atol=1e-6), f"output mismatch on {rank=}"
    assert torch.allclose(tp_input.grad, ref_input_grad, atol=1e-6), f"input_grad mismatch on {rank=}"
    print(end=f"Tests passed ({rank=})\n", flush=True); dist.barrier()

# fun fact: 90% of the code above was generated by grok-3 for prompt "Please rewrite the following code using torch.distributed.tensor ```python <paste MLP code here>```"
# the remaining 10% are nasty bugfixes that took 99% of assignment preparation time. Do not trust the shogoths yet :)

Overwriting tensor_parallel_mlp_dtensor.py


In [ ]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_mlp_dtensor.py

/usr/local/lib/python3.11/dist-packages/torch/distributed/tensor/_random.py:45: UserWarning: DTensor random operators may not have complete support on cpu device mesh
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/distributed/tensor/_random.py:45: UserWarning: DTensor random operators may not have complete support on cpu device mesh
  warnings.warn(
name='0.weight',	type=<class 'torch.Tensor'>	global shape=torch.Size([1024]),	local shape=torch.Size([1024])
name='1.gate_proj.weight',	type=<class 'torch.distributed.tensor.DTensor'>	global shape=torch.Size([4096, 1024]),	local shape=torch.Size([2048, 1024])
name='1.up_proj.weight',	type=<class 'torch.distributed.tensor.DTensor'>	global shape=torch.Size([4096, 1024]),	local shape=torch.Size([2048, 1024])
name='1.down_proj.weight',	type=<class 'torch.distributed.tensor.DTensor'>	global shape=torch.Size([1024, 4096]),	local shape=torch.Size([1024, 2048])

Reference outputs (rank=0): tensor([[ 0.0102,  0.0432, -0.0467,  ...,  

In [ ]:
%%writefile tensor_parallel_llama_dtensor.py
"""
Task 3: DTensor-based Tensor Parallel Llama Inference + Speed Comparison

We re-implement Task 2 (manual all_reduce tensor parallel) using torch.distributed.tensor (DTensor) APIs.
Goals:
 1. Shard transformer layer parameters (attention q/k/v col-wise, o row-wise; MLP gate/up col-wise, down row-wise) using tp.* helpers.
 2. Replicate embeddings, final norm, lm_head (simpler per assignment).
 3. Run correctness checks against full reference model (logits + embedding gradient) on rank 0.
 4. Provide a simple timing comparison vs manual implementation (user should have run Task 2 first). We measure forward+backward latency for a few warmup + measured iterations.
 5. Run 10-step greedy generation with DTensor model (recompute, no KV cache).

Assumptions:
 - world_size divides num_heads and intermediate_size.
 - Backend: gloo (CPU) or nccl (GPU). Use bf16/fp16 if needed for memory.
 - PyTorch version supports distributed.tensor.parallel (>=2.3+ typically).

How to run (CPU example, 2 ranks):
   OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_llama_dtensor.py --model unsloth/Llama-3.2-1B --backend gloo --dtype fp32
GPU example:
   torchrun --nproc_per_node 2 tensor_parallel_llama_dtensor.py --backend nccl --dtype bf16
"""
import argparse, time, math
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.nn.functional as F
from transformers import LlamaForCausalLM, AutoTokenizer

from torch.distributed.device_mesh import init_device_mesh
import torch.distributed.tensor.parallel as tp

# ---------------- Argument parsing ----------------

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--model', default='unsloth/Llama-3.2-1B')
    p.add_argument('--backend', default='gloo')
    p.add_argument('--dtype', default='fp32', choices=['fp32','bf16','fp16'])
    p.add_argument('--gen-steps', type=int, default=10)
    p.add_argument('--warmup', type=int, default=2)
    p.add_argument('--iters', type=int, default=5)
    return p.parse_args()

def str_to_dtype(s):
    if s=='fp32': return torch.float32
    if s=='bf16': return torch.bfloat16
    if s=='fp16': return torch.float16
    raise ValueError(s)

# ------------- DTensor parallelization helper -------------

def parallelize_llama_layers(ref_model: LlamaForCausalLM, device_mesh):
    """Return a module whose transformer layers are tensor-parallel via DTensor.
    Strategy (mirrors manual TP):
      gate_proj, up_proj -> ColwiseParallel
      down_proj -> RowwiseParallel
      q_proj, k_proj, v_proj -> ColwiseParallel
      o_proj -> RowwiseParallel
    RMSNorms remain replicated.
    """
    # We apply plan onto the submodules referencing original modules (in-place re-sharding)
    parallelize_plan = {}
    for layer_idx, layer in enumerate(ref_model.model.layers):
        prefix = f"model.layers.{layer_idx}"
        # attention
        parallelize_plan[f"{prefix}.self_attn.q_proj"] = tp.ColwiseParallel()
        parallelize_plan[f"{prefix}.self_attn.k_proj"] = tp.ColwiseParallel()
        parallelize_plan[f"{prefix}.self_attn.v_proj"] = tp.ColwiseParallel()
        parallelize_plan[f"{prefix}.self_attn.o_proj"] = tp.RowwiseParallel()
        # mlp
        parallelize_plan[f"{prefix}.mlp.gate_proj"] = tp.ColwiseParallel()
        parallelize_plan[f"{prefix}.mlp.up_proj"] = tp.ColwiseParallel()
        parallelize_plan[f"{prefix}.mlp.down_proj"] = tp.RowwiseParallel()
    # NOTE: embed_tokens, final norm, lm_head left replicated
    tp_module = tp.parallelize_module(ref_model, device_mesh, parallelize_plan=parallelize_plan)
    return tp_module

# ------------- Timing utilities -------------

def measure_latency(model, input_ids, steps_fwd_bwd=1, backward=True):
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    start = time.perf_counter()
    for _ in range(steps_fwd_bwd):
        out = model(input_ids).logits
        loss = out[:, -1].sum()
        if backward:
            loss.backward()
            for p in model.parameters():
                if p.grad is not None:
                    p.grad.zero_()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    return (time.perf_counter() - start) / steps_fwd_bwd

# ------------- Generation (no KV cache) -------------
@torch.no_grad()
def generate(model, tokenizer, input_ids, steps=10):
    for _ in range(steps):
        logits = model(input_ids).logits
        next_token = logits[:, -1].argmax(-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_token], dim=1)
        if dist.get_rank() == 0:
            print(tokenizer.decode(next_token[0]), end="", flush=True)
    if dist.get_rank() == 0:
        print()
    return input_ids

# ------------- Main -------------

def main():
    args = parse_args()
    dist.init_process_group(args.backend)
    rank = dist.get_rank(); world_size = dist.get_world_size()
    dtype = str_to_dtype(args.dtype)
    torch.manual_seed(1234)

    if rank == 0:
        print(f"[Init] Loading reference model: {args.model} (dtype={dtype}, world_size={world_size})")
    # Sequential load to save memory (barrier per rank)
    model = None; tokenizer = None
    for r in range(world_size):
        dist.barrier()
        if r != rank: continue
        model = LlamaForCausalLM.from_pretrained(args.model, torch_dtype=dtype)
        tokenizer = AutoTokenizer.from_pretrained(args.model)
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id
        if rank == 0:
            print("[Init] Reference model loaded.")

    dist.barrier()

    # Create device mesh (cpu or cuda)
    device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
    device_mesh = init_device_mesh(device_type=device_type, mesh_shape=(world_size,))

    # Parallelize in-place
    if rank == 0:
        print("[TP] Applying DTensor sharding plan to transformer layers ...")
    tp_model = parallelize_llama_layers(model, device_mesh)

    # Prepare input
    prompt = "A quick brown fox"
    input_ids = tokenizer(prompt, return_tensors='pt')["input_ids"].to(device_type)
    input_ids_ref = input_ids.clone().requires_grad_(True)

    # Reference forward/backward (rank 0 only) for correctness
    if rank == 0:
        ref_out = model(input_ids_ref).logits
        ref_loss = ref_out[:, -1].sum(); ref_loss.backward()
        ref_emb_grad = model.model.embed_tokens.weight.grad.clone()
        ref_logits_detached = ref_out.detach()
    else:
        ref_logits_detached = torch.zeros((1, input_ids.shape[1], model.lm_head.out_features), dtype=dtype, device=device_type)
        ref_emb_grad = torch.zeros_like(model.model.embed_tokens.weight)

    # Broadcast reference tensors to all ranks for comparison
    dist.broadcast(ref_logits_detached, src=0)
    dist.broadcast(ref_emb_grad, src=0)

    # DTensor forward/backward
    dt_out = tp_model(input_ids).logits
    dt_loss = dt_out[:, -1].sum(); dt_loss.backward()

    # Embedding grad (replicated) should match
    emb_grad = tp_model.model.embed_tokens.weight.grad
    # All-reduce grad to mirror reference accumulation semantics (DTensor may have already placed grads properly; safe to sum)
    dist.all_reduce(emb_grad, op=dist.ReduceOp.SUM)

    if rank == 0:
        logits_diff = (dt_out - ref_logits_detached).abs().max().item()
        grad_diff = (emb_grad - ref_emb_grad).abs().max().item()
        print(f"[Check] logits max|diff| = {logits_diff:.3e}")
        print(f"[Check] embed grad max|diff| = {grad_diff:.3e}")
        assert logits_diff < 1e-4, "Logits mismatch"
        assert grad_diff < 1e-4, "Embedding grad mismatch"
        print("[Check] Correctness PASSED.")

    # Timing benchmark
    dist.barrier()
    if rank == 0:
        print("[Bench] Measuring DTensor forward+backward latency ...")
    # Warmup
    for _ in range(args.warmup):
        _ = tp_model(input_ids).logits[:, -1].sum().backward()
        for p in tp_model.parameters():
            if p.grad is not None: p.grad.zero_()
    dist.barrier()
    t_dt = measure_latency(tp_model, input_ids, steps_fwd_bwd=args.iters, backward=True)

    # Dummy manual baseline placeholder (user may record from Task 2 run)
    # We log dtensor time; user can copy manual time from Task2 run to compare.
    if rank == 0:
        print(f"[Bench] DTensor avg fwd+bwd time over {args.iters} iters: {t_dt:.4f}s (record manual TP separately)")

    # Generation
    if rank == 0:
        print("[Gen] DTensor generation:", end=" ")
        print(prompt, end="")
    generate(tp_model, tokenizer, input_ids, steps=args.gen_steps)

    if rank == 0:
        print("Task 3 DTensor implementation complete.")

    dist.barrier()
    del model

if __name__ == '__main__':
    main()


In [ ]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_llama_dtensor.py --backend gloo --dtype fp32 --gen-steps 10 --warmup 1 --iters 3
# GPU example:
# torchrun --nproc_per_node 2 tensor_parallel_llama_dtensor.py --backend nccl --dtype bf16 --gen-steps 10 --warmup 2 --iters 5

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```



### Sequence Parallelism with Ulysses


Now let's parallelize the other way - across the sequence dimension. To showcase why this is necessary, our main task will be to parallelize LLM fine-tuning over a very long sequence. The way you do this, of course, is through Sequence Parallelism. You can implement naive [sequence parallelism](https://arxiv.org/abs/2205.05198), similar to [DeepSpeed Ulysses](https://arxiv.org/pdf/2309.14509) (n.b.: not the first work to do this).

![figure-from-paper](https://ar5iv.labs.arxiv.org/html/2309.14509/assets/figs/image3.png)


Here's the short version:
- All weights are replicated between ranks (optionally: FSDP)
- Each rank holds a subset of sequence tokens
- Embeddings, logits, normalizations, MLP all apply independently to token shards
- The multi-head attention is the only layer that gets special treatment
    - First, apply QKV projections to local tokens, as in data-parallel training;
    - Then re-shard so that each rank holds a **subset of heads** across **all tokens**;
    - Compute the attention ''core'' (RoPE and F.scaled_dot_product_attention) for its chunk of heads independently;
    - Re-shard outputs again so that each rank concatenates **all heads**, but only for its **subset of tokens**;
    - Apply the output ("O") projection to your local tokens again.
- This approach *may* be combined with tensor parallelism, but this is an advanced technique that you don't have to implement.


__You have a choice__ between two options on how to implement it: either manually with torch.distributed like in task 2, or using the DTensor route like in task 3. We provide some tips for both tasks.


**Option A. with raw `torch.distirbuted`:**
- Use [`dist.all_to_all`](https://pytorch.org/docs/stable/distributed.html#torch.distributed.all_to_all) to switch between per-token and per-head sharding without materializing the full tensor on any device;
- Wrap the model with [`DistributedDataParallel`](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html) or [`FullyShardedDataParallel`](https://pytorch.org/docs/stable/fsdp.html) so that fine-tuning synchronizes trainable parameters. Note that using FSDP for parameter-efficient fine-tuning can be tricky: we recommend you either wrap **trainable modules** with separate FSDP sub-instances via auto_wrap_policy - or simply use DDP instead of FSDP.

**Option B. with `DTensor`:**
- We recommend you first skim the official [tutorial](https://pytorch.org/tutorials/intermediate/TP_tutorial.html) on applying Tensor Parallelism (sic.) - or browse the [TorchTitan's version](https://github.com/pytorch/torchtitan/blob/82afc842e303e49d1a137fc7ea48291a57f72d5d/torchtitan/models/llama/parallelize_llama.py) of it.
- Note that there is a [`SequenceParallel`](https://pytorch.org/docs/stable/distributed.tensor.parallel.html#torch.distributed.tensor.parallel.SequenceParallel) class in torch.distributed.tensor.parallel` - **however, it does not magick the sequence parallelism for you** - it is only meant for small layers (e.g. normalization). You still need to do the sharding in self-attention!

For the sake of formality, here's an example script you need to parallelize:

In [ ]:
import torch
import transformers
import peft
MODEL_NAME = "unsloth/Llama-3.2-1B"  # for testing (but not grading!), you may want to use Maykeye/TinyLLama-v0
SEQUENCE_LENGTH = 128                # IMPORTANT!!! you need to increase this parameter! Look for the maximum sequence length on one and multiple GPUs

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = transformers.LlamaForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16).to(device)

for param in model.parameters():
  param.required_grad = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

model = peft.get_peft_model(model, peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=32))
assert any(param.requires_grad for param in model.parameters()), "No trainable parameters - did you enable PEFT?"

!wget -q https://www.gutenberg.org/cache/epub/4300/pg4300.txt -O ulysses.txt  # ... or use any other text of your choosing
input_ids = tokenizer(open("ulysses.txt").read(), return_tensors='pt')['input_ids']
print(f"Cropping {input_ids.shape[1]=} to {SEQUENCE_LENGTH} tokens")
input_ids, labels = input_ids[:, :SEQUENCE_LENGTH], input_ids[:, 1:SEQUENCE_LENGTH + 1]

trainable_parameters = {p for p in model.parameters() if p.requires_grad}
print(f"Parameters: {sum(map(torch.Tensor.numel, trainable_parameters))} trainable / {sum(map(torch.Tensor.numel, model.parameters()))} total")
opt = torch.optim.Adam(trainable_parameters)
for i in range(10):
  loss = model(input_ids=input_ids.to(device), labels=labels.to(device)).loss
  opt.zero_grad()
  loss.backward()
  opt.step()
  print(f"{i=}\t{loss.item()=}")

# pro tip: delete the model or restart session to free RAM for the TP experiments

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (397368 > 131072). Running this sequence through the model will result in indexing errors


Cropping input_ids.shape[1]=397368 to 128 tokens
Parameters: 65536 trainable / 1235879936 total
i=0	loss.item()=8.582364082336426
i=1	loss.item()=8.413138389587402
i=2	loss.item()=8.251019477844238
i=3	loss.item()=8.113922119140625
i=4	loss.item()=8.001195907592773
i=5	loss.item()=7.875072002410889
i=6	loss.item()=7.784201145172119
i=7	loss.item()=7.691455364227295
i=8	loss.item()=7.621612548828125
i=9	loss.item()=7.547983169555664


__Task 4 (1 point):__ before you do training, let's first parallelize a single forward pass. Implement sharding with the same interface you used in tasks 2 (or 3 if you use DTensor), but this time, parallelize across the sequence dimension. Note: if you are running out of (V)RAM, load the 1B model in half precision and disable gradients for all weights except the first (few) layers.



In [ ]:
<<A whole lot of your code here>>

In [ ]:
<<... and a dedicated cell to show off that it works>>

__Task 5 (1 point):__ Now use the script above to parallelize the entire training run. You are free to use other fine-tuning methods (e.g. LoRA or even full fine-tuning), as long as you can demonstrate that the loss goes down.

**Make sure you increase SEQUENCE_LENGTH as much as possible!** Even on a single GPU, you should be able to go into thousands, if not tens of thousands of tokens - and report the maximum sequence length with one and with multiple GPUs respectively.

If you don't have access to multiple GPUs, you may optionally submit a version that does training on a single GPU, but computes attention heads sequentially with gradient checkpointing - but if you do, please announce that you are using this option in bold, capital letters, so that the grader will notice it.

In [ ]:
<<A whole lot of your code here>>

In [ ]:
<<... and a dedicated cell to show off that it works>>

```

```

```

```

```

```


### Optional: bonus tasks

There are many routes to further improve the training/inference code. You may (but you don't have to) implement any combination of them for bonus points.

However, please not that the total points for this week's entire assignment (part 1 & 2) are **capped at 14**.

__Bonus task: parallel key-value caching (1 point).__ In tasks 2 and 3, you implement tensor parallelism for attention forward pass and perform inference with re-computation. However, real world inference engines use [KV caching](https://huggingface.co/docs/transformers/main/en/kv_cache) - keeping key and value caches from past tokens and only processing the new token each time.

For this task, you will have to implement this type of parallelism for either torch.distributed or DTensor implementation of attention $-$ simply cache the heads already assigned to each rank. To get the grade, you will need to demonstrate that the model generates a sensible text with any cache (via past_key_values=).

__Bonus task: pipeline parallelism (1-2 points):__ In tasks 1-3, you've implemented symmetric model parallelism, aka Tensor Parallelism. However, there is another way to partition model parameters $-$ assign entire layers to each rank and run them in a pipeline. This can be faster, especially if you are running

For 1 point, check out [torch.distributed.pipelinging](https://pytorch.org/docs/stable/distributed.pipelining.html), [DeepSpeed pipelining](https://deepspeed.readthedocs.io/en/latest/pipeline.html) or [torchgpipe](https://github.com/kakaobrain/torchgpipe) and demonstrate that you can run or fine-tune a model that would not fit into a single GPU (you will need multuple devices for this!).

For 2 points, compare different pipelining schedules in terms of training throughput: use GPipe as a baseline and try ScheduleInterleaved1F1B (or a more advanced pipeline of your choosing).

__Bonus task: better sequence parallelism (2 points).__ In tasks 4 and 5, you implemented basic sequence parallelism. However, there are multiple ways you can improve that technique for further memory savings or better device utilization.

For 1 point, implement combined tensor + sequence parallelism and compare results with naive sequence parallelism.

For 2 points, implement [Ring Attention](https://arxiv.org/abs/2310.01889) *or* integrate computation-communication overlap from [FLUX](https://arxiv.org/abs/2406.06858) and measure the speed and memory trade-offs.